In [1]:
import fmatoolbox as fma
from Detector_final import PyProcessor
from detector_simulator import MockProcessor
import pathlib
froot = pathlib.Path.cwd() / 'CONFIGS' / 'SimulateOpenEphys'
batch_file = '/mnt/hubel-data-103/Pietro/InfraSlowNRPaper/Data/IS_intervals.batch'

In [2]:
import pathlib
import xml.etree.ElementTree
def loadParam(session,names,dtypes=None):

    session = pathlib.Path(session)

    # load parameters n_channels from .xml
    tree = xml.etree.ElementTree.parse(session.with_suffix('.xml'))
    root = tree.getroot()

    parameters = {}
    for name in names:
        parameters[name] = root.find(f'.//{name}').text
        if dtypes == 'int':
            parameters[name] = int(parameters[name])

    return parameters

In [3]:
# test on one session
session = fma.data.readBatchFile(batch_file)[0][9]
print(session)
parameters = loadParam(session,['samplingRate','nChannels'],dtypes='int')
packet_size = int(parameters['samplingRate'] * 0.12) # 120 ms

/mnt/hubel-data-131/perceval/Rat003_20231224/Rat003_20231224.xml


In [4]:
py_proc = PyProcessor(processor=MockProcessor(),num_channels=parameters['nChannels'],sample_rate=parameters['samplingRate'],config_path=froot / 'config_test_simOE.py')

Num Channels: 142| Sampling Rate: 20000
[OE] Config chargée depuis : /media/data-103/Pietro/InfraSlowRhythmLiveDetector/Code/Python/FINAL_Detector/CONFIGS/SimulateOpenEphys/config_test_simOE.py
[OE] 10 détecteur(s) : ['ref_v1', '5-1', '5-1.5', '3-0.3', '3-0.6', '3-0.9', '7-0.7', '7-1.4', '7-2.1', '7-3.5']


In [5]:
data, t = fma.data.loadWideband(session,intervals=[0,10])
data = data.T

In [6]:
ttl = {2: (9,True)} # iteration n: TTL line

In [7]:
for i in range(data.shape[1] // packet_size):

    if i in ttl:
        py_proc.handle_ttl_event(source_node=None,channel=None,sample_number=None,line=ttl[i][0],state=ttl[i][1])

    py_proc.process(data[:, packet_size*i: packet_size*(i+1)])

added python event: 3,True
[OE] Phase 1 démarrée pour 10 détecteurs.
added python event: 7,False
added python event: 4,False
added python event: 1,False
added python event: 12,False
added python event: 6,False
added python event: 13,False
